# 01 — Demo rò rỉ dữ liệu (data leakage): cột `duration`

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

RANDOM_STATE = 42

## 1. Đọc dữ liệu

In [3]:
df = pd.read_csv('../dataset/bank-additional-full.csv', sep=';')
print(df.shape)
df.head()
df['y'].value_counts(normalize=True)

(41188, 21)


y
no     0.887346
yes    0.112654
Name: proportion, dtype: float64

## 2. Chạy 2 phiên bản: CÓ và KHÔNG có `duration`

In [5]:
cat_cols = ['job','marital','education','default','housing','loan',
            'contact','month','day_of_week','poutcome']
num_cols_common = ['age','campaign','pdays','previous',
                    'emp.var.rate','cons.price.idx','cons.conf.idx',
                    'euribor3m','nr.employed']

y = (df['y'] == 'yes').astype(int)

def quick_auc(num_cols, n_estimators=200):
    X = df[cat_cols + num_cols]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

    pre = ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
    ], remainder='passthrough')

    pipe = Pipeline([
        ('pre', pre),
        ('rf', RandomForestClassifier(n_estimators=n_estimators, max_features='sqrt',
                                       class_weight='balanced_subsample',
                                       n_jobs=-1, random_state=RANDOM_STATE))
    ])
    pipe.fit(X_train, y_train)
    proba = pipe.predict_proba(X_test)[:, 1]
    return roc_auc_score(y_test, proba)
auc_with = quick_auc(num_cols_common + ['duration'])
auc_without = quick_auc(num_cols_common)

print(f'AUC CÓ duration     : {auc_with:.4f}')
print(f'AUC KHÔNG có duration: {auc_without:.4f}')
print(f'Chênh lệch           : {auc_with - auc_without:+.4f}')

AUC CÓ duration     : 0.9473
AUC KHÔNG có duration: 0.7834
Chênh lệch           : +0.1639
